In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
import os

audio_dir = '/content/drive/MyDrive/dialect_data/audio'
label_dir = '/content/drive/MyDrive/dialect_data/labels'

audio_files = os.listdir(audio_dir)
label_files = os.listdir(label_dir)

print(f"오디오 파일 수: {len(audio_files)}")
print(f"라벨 파일 수: {len(label_files)}")
print(audio_files[:3])

오디오 파일 수: 35
라벨 파일 수: 35
['st_set2_collectorgw185_speakergw1744_71_4.wav', 'st_set2_collectorgw185_speakergw1744_67_5.wav', 'st_set2_collectorgw185_speakergw1744_73_6.wav']


In [11]:
!pip install transformers datasets librosa soundfile torch torchaudio -q

In [12]:
import json
import os
import librosa
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration

# Whisper 모델 로드
processor = WhisperProcessor.from_pretrained("openai/whisper-small")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")

print("모델 로드 완료!")

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

모델 로드 완료!


In [13]:
import json
import numpy as np

def load_data(audio_dir, label_dir):
    data = []
    audio_files = sorted(os.listdir(audio_dir))

    for audio_file in audio_files:
        if not audio_file.endswith('.wav'):
            continue

        # 오디오 로드
        audio_path = os.path.join(audio_dir, audio_file)
        speech, sr = librosa.load(audio_path, sr=16000)

        # 라벨 찾기 (같은 이름의 json)
        label_file = audio_file.replace('.wav', '.json')
        label_path = os.path.join(label_dir, label_file)

        if not os.path.exists(label_path):
            continue

        with open(label_path, 'r', encoding='utf-8') as f:
            label_data = json.load(f)

        # 표준어 텍스트 추출
        text = label_data['transcription']['standard']

        data.append({'audio': speech, 'text': text})

    print(f"로드된 데이터 수: {len(data)}")
    return data

dataset = load_data(audio_dir, label_dir)
print(f"샘플 텍스트: {dataset[0]['text']}")

로드된 데이터 수: 35
샘플 텍스트: 아까 내가 사이즈 먹고 그럴 때부터 분명히 넉넉할  거라고 했는데 나한테 어림도 없으니까 하나 더 큰 거 주세요


In [14]:
from torch.utils.data import Dataset, DataLoader
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
import torch

class DialectDataset(Dataset):
    def __init__(self, data, processor):
        self.data = data
        self.processor = processor

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        inputs = self.processor(
            item['audio'],
            sampling_rate=16000,
            return_tensors='pt'
        )
        input_features = inputs.input_features[0]

        labels = self.processor.tokenizer(
            item['text'],
            return_tensors='pt'
        ).input_ids[0]

        return {
            'input_features': input_features,
            'labels': labels
        }

train_dataset = DialectDataset(dataset, processor)
print(f"데이터셋 크기: {len(train_dataset)}")
print("데이터셋 준비 완료!")

데이터셋 크기: 35
데이터셋 준비 완료!


In [16]:
!pip install evaluate -q

In [15]:
from transformers import DataCollatorForSeq2Seq
import evaluate

def collate_fn(batch):
    input_features = torch.stack([item['input_features'] for item in batch])

    label_list = [item['labels'] for item in batch]
    max_len = max(l.size(0) for l in label_list)
    padded_labels = torch.full((len(label_list), max_len), -100, dtype=torch.long)
    for i, label in enumerate(label_list):
        padded_labels[i, :label.size(0)] = label

    return {
        'input_features': input_features,
        'labels': padded_labels
    }

training_args = Seq2SeqTrainingArguments(
    output_dir='/content/drive/MyDrive/dialect_data/model',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    learning_rate=1e-5,
    save_steps=50,
    logging_steps=10,
    predict_with_generate=True,
    fp16=False,
    report_to='none'
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=collate_fn,
    processing_class=processor.feature_extractor,
)

print("학습 시작!")
trainer.train()
print("학습 완료!")

학습 시작!


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
10,3.156009
20,2.211901
30,1.455460
40,1.077313
50,0.955472


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

학습 완료!


In [20]:
model.save_pretrained('/content/drive/MyDrive/dialect_data/model/whisper-dialect')
processor.save_pretrained('/content/drive/MyDrive/dialect_data/model/whisper-dialect')
print("모델 저장 완료!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

모델 저장 완료!


In [22]:
dataset = load_data(audio_dir, label_dir)

로드된 데이터 수: 50


In [23]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration

model = WhisperForConditionalGeneration.from_pretrained('/content/drive/MyDrive/dialect_data/model/whisper-dialect')
processor = WhisperProcessor.from_pretrained('/content/drive/MyDrive/dialect_data/model/whisper-dialect')
print("파인튜닝 모델 로드 완료!")

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

파인튜닝 모델 로드 완료!


In [24]:
train_dataset = DialectDataset(dataset, processor)
print(f"데이터셋 크기: {len(train_dataset)}")

training_args = Seq2SeqTrainingArguments(
    output_dir='/content/drive/MyDrive/dialect_data/model',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    learning_rate=1e-5,
    save_steps=50,
    logging_steps=10,
    predict_with_generate=True,
    fp16=False,
    report_to='none'
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=collate_fn,
    processing_class=processor.feature_extractor,
)

print("추가 학습 시작!")
trainer.train()
print("추가 학습 완료!")

데이터셋 크기: 50
추가 학습 시작!


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
10,0.772553
20,0.584401
30,0.451722
40,0.317185
50,0.307073
60,0.225240
70,0.238958


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

추가 학습 완료!


In [25]:
model.save_pretrained('/content/drive/MyDrive/dialect_data/model/whisper-dialect')
processor.save_pretrained('/content/drive/MyDrive/dialect_data/model/whisper-dialect')
print("모델 저장 완료!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

모델 저장 완료!


In [26]:
forced_decoder_ids = processor.get_decoder_prompt_ids(language="korean", task="transcribe")
model.config.forced_decoder_ids = forced_decoder_ids

test_audio = dataset[0]['audio']
inputs = processor(test_audio, sampling_rate=16000, return_tensors='pt')

with torch.no_grad():
    predicted_ids = model.generate(
        inputs.input_features,
        forced_decoder_ids=forced_decoder_ids
    )

transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)
print(f"정답: {dataset[0]['text']}")
print(f"예측: {transcription[0]}")

정답: 저는 오 녀 일 남인데 언니는 창원에 살고 동생들은 서울에 살고 막내 동생은 호주에 살고 있어요 다 흩어져 있어서 자주 보지는 못해도 가끔 보면 아주 반가워요
예측: 저는 오 녀 일 남인데 언니는 창원에 살고 동생들은 서울에 살고 막내 동생은 호주에 살고 있어요 다 흩어져 있어서 자주 보지는 못해도 가끔 보면 아주 반가워요
